<a href="https://colab.research.google.com/github/Vigneshvar16/NLP_DSA/blob/main/NLP_ASSIGNMENT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
hospital_nlp.py  -  DSA03 Natural Language Processing, Assignment 1 (CO1)
Morphology-aware text normalisation and search layer for a hospital record system.

Run:   python hospital_nlp.py            (needs: pip install nltk ; matplotlib optional, for the plot)
Does:  builds a documented SYNTHETIC corpus (520 records, 4 record types), then runs
       Tasks 1-12: regex entity patterns, tokenizer/normaliser, DFA/NFA (Thompson, subset
       construction, minimisation, state elimination), a finite-state morphological transducer
       (lexicon FST + orthographic rule cascade, bidirectional), Porter stemmer comparison,
       retrieval experiment, and the Day-4 noisy-input redesign (OCR + Romanised code-mixing).
Out:   results.json (all numbers used in the report) and coverage.png.   Libraries: stdlib + NLTK.
"""
import re, random, json, time, string, itertools, functools, math, sys
from collections import Counter, defaultdict
from nltk.stem import PorterStemmer

R = random.Random(7)
PS = PorterStemmer(PorterStemmer.ORIGINAL_ALGORITHM)
OUT = {}

# =====================================================================================
# 1. LEXICON DATA  (stems grouped semantically: the groups drive both lexicon and corpus)
# =====================================================================================
SYMP = "fever cough headache pain rash swelling nausea fatigue chill cramp itch tremor bruise seizure palpitation wheeze spasm"
BODY = "body kidney liver lung heart brain bone joint muscle nerve vein artery spine skull rib hip knee ankle shoulder elbow wrist finger thumb toe eye ear nose throat tongue skin abdomen chest valve vertebra foot tooth stomach"
COND = "fracture wound lesion tumour cyst ulcer abscess infection allergy diabetes stroke clot asthma sepsis hernia polyp cataract fistula embolus thrombus bacterium fungus metastasis stenosis prognosis diagnosis anastomosis"
PROC = "procedure surgery biopsy incision transplant graft stent implant dialysis bypass fixation scan shunt pacemaker plate screw rod pin wire mesh"
ITEM = "stitch box tablet capsule injection infusion dose bandage dressing plaster cast splint crutch catheter tube cannula needle syringe suture drain mask monitor ventilator stretcher wheelchair trolley sample specimen culture film chart bed ward line tray"
PERS = "patient doctor nurse surgeon physician therapist pharmacist attendant relative"
ABS = ("history complaint finding sign course stay visit review plan advice appointment referral consent risk benefit outcome recovery relapse remission "
       "result report note record test diet fluid blood urine stool sputum oxygen glucose salt iron calcium potassium sodium protein enzyme hormone insulin "
       "steroid vitamin supplement antibiotic analgesic sedative anaesthetic pressure pulse rate rhythm breath tension norm region nutrition emotion function "
       "condition origin biotic coagulant pyretic emetic")
VSURG = "operate excise resect suture drain remove insert close ligate clamp perforate dilate rupture transfuse intubate ventilate sedate"
VMED = "prescribe administer inject infuse increase decrease reduce taper titrate discontinue continue start stop repeat advise recommend"
VOBS = "observe examine monitor check measure assess review test screen detect record note report chart diagnose suspect"
VCARE = "dress wash clean wrap bandage splint mobilise walk rest flush instruct counsel educate prepare support"
VSYM = "cough faint collapse sweat shiver tremble ache worsen deteriorate improve recover heal respond relapse nauseate clot vomit"
VOTH = ("agree dye admit discharge refer complain present develop occur plan scan resolve complete perform apply order wait call inform consent supervise protect "
        "restrict obstruct disrupt displace replace involve impair attach combine determine consult transform avoid cure study act hydrate prevent narrow "
        "block sprain strain poison infect inflame irritate limit offer suffer enter edit open visit manage notice fracture stabilise tolerate treat culture")
VIRR_L = "give take see go have break bleed feed hold keep lose get leave send sit stand tell swell fall drink eat sleep wake hurt find bring say come run put make think cut bite sting bend do be"
AG = "mild severe high low large small early late heavy dry weak strong sick thin thick slow short long old young hot cold warm wet red big deep slight fine"
AN = ("acute chronic stable critical benign malignant fatal toxic allergic febrile painful tender swollen normal abnormal elevated moderate infectious viral "
      "renal cardiac hepatic pulmonary venous muscular cranial cutaneous vascular conscious sterile active sensitive mobile fertile intact clinical medical "
      "surgical obvious dizzy drowsy tired numb stiff ill rapid male female left right")
SEX = "male female"
FUNC = ("the a an of in on at to for with and or but not no by from after before during under over about as if then than that this these those he she it they "
        "his her their its there which who when while without within into per via also only both each all any some other more most less very too well still "
        "again already today daily weekly once twice thrice three four five six seven eight nine ten one two")
LIT_EXTRA = "was were is are been has had have"            # literals recognised through the irregular verbs be/have
EXC = set("visit edit open develop limit monitor vomit offer suffer enter order shiver recover taper worsen poison stomach manage notice dye".split())
MASS = set("diabetes sepsis blood urine oxygen glucose iron calcium potassium sodium protein insulin fluid salt nutrition history dialysis".split())
IRR_V = {"be": {"3SG": ["is"], "PAST": ["was", "were"], "PASTP": ["been"], "PRESP": ["being"]}, "go": {"3SG": ["goes"], "PAST": ["went"], "PASTP": ["gone"]},
         "give": {"PAST": ["gave"], "PASTP": ["given"]}, "take": {"PAST": ["took"], "PASTP": ["taken"]}, "see": {"PAST": ["saw"], "PASTP": ["seen"]},
         "do": {"3SG": ["does"], "PAST": ["did"], "PASTP": ["done"]}, "have": {"3SG": ["has"], "PAST": ["had"], "PASTP": ["had"]},
         "break": {"PAST": ["broke"], "PASTP": ["broken"]}, "bleed": {"PAST": ["bled"], "PASTP": ["bled"]}, "feed": {"PAST": ["fed"], "PASTP": ["fed"]},
         "hold": {"PAST": ["held"], "PASTP": ["held"]}, "keep": {"PAST": ["kept"], "PASTP": ["kept"]}, "lose": {"PAST": ["lost"], "PASTP": ["lost"]},
         "get": {"PAST": ["got"], "PASTP": ["gotten"]}, "leave": {"PAST": ["left"], "PASTP": ["left"]}, "send": {"PAST": ["sent"], "PASTP": ["sent"]},
         "sit": {"PAST": ["sat"], "PASTP": ["sat"]}, "stand": {"PAST": ["stood"], "PASTP": ["stood"]}, "tell": {"PAST": ["told"], "PASTP": ["told"]},
         "swell": {"PASTP": ["swollen"]}, "fall": {"PAST": ["fell"], "PASTP": ["fallen"]}, "drink": {"PAST": ["drank"], "PASTP": ["drunk"]},
         "eat": {"PAST": ["ate"], "PASTP": ["eaten"]}, "sleep": {"PAST": ["slept"], "PASTP": ["slept"]}, "wake": {"PAST": ["woke"], "PASTP": ["woken"]},
         "hurt": {"PAST": ["hurt"], "PASTP": ["hurt"]}, "find": {"PAST": ["found"], "PASTP": ["found"]}, "bring": {"PAST": ["brought"], "PASTP": ["brought"]},
         "say": {"PAST": ["said"], "PASTP": ["said"]}, "come": {"PAST": ["came"], "PASTP": ["come"]}, "run": {"PAST": ["ran"], "PASTP": ["run"]},
         "put": {"PAST": ["put"], "PASTP": ["put"]}, "make": {"PAST": ["made"], "PASTP": ["made"]}, "think": {"PAST": ["thought"], "PASTP": ["thought"]},
         "cut": {"PAST": ["cut"], "PASTP": ["cut"]}, "bite": {"PAST": ["bit"], "PASTP": ["bitten"]}, "sting": {"PAST": ["stung"], "PASTP": ["stung"]},
         "bend": {"PAST": ["bent"], "PASTP": ["bent"]}}
IRR_N = {"diagnosis": "diagnoses", "prognosis": "prognoses", "stenosis": "stenoses", "anastomosis": "anastomoses", "metastasis": "metastases",
         "vertebra": "vertebrae", "bacterium": "bacteria", "fungus": "fungi", "thrombus": "thrombi", "embolus": "emboli", "foot": "feet", "tooth": "teeth",
         "dialysis": "dialyses"}
IRR_A = {"good": {"COMP": ["better"], "SUP": ["best"]}, "bad": {"COMP": ["worse"], "SUP": ["worst"]}}
# derivational suffixes: tag, surface, category-in, category-out, whitelisted stems (the finite-state layer cannot know which derivations exist)
DER = [("ITIS", "itis", "CF", "N", "nephr hepat gastr arthr dermat bronch tonsill appendic cholecyst mening encephal myel laryng pharyng rhin sinus cyst pancreat col gloss phleb ot neur thyroid"),
       ("ECTOMY", "ectomy", "CF", "N", "nephr append cholecyst hyster mast col splen thyroid tonsill gastr laryng prostat pancreat"),
       ("OTOMY", "otomy", "CF", "N", "lapar thorac crani col gastr nephr"),
       ("OLOGY", "ology", "CF", "N", "cardi neur path radi onc ur hemat nephr dermat immun ophthalm rheumat"),
       ("OSIS", "osis", "CF", "N", "fibr thromb necr scler cirrh tubercul sten neur nephr arthr"),
       ("AEMIA", "aemia", "CF", "N", "an leuk glyc bacter isch ur tox"),
       ("IC", "ic", "CF", "ADJ", "hepat gastr anaem diabet metabol sept alcohol nephr"),
       ("AL", "al", "N|CF", "ADJ", "bacteri norm procedure region nutrition emotion function condition origin"),
       ("ATION", "ation", "V", "N", "observe examine inform consult prepare combine determine transform"),
       ("ION", "ion", "V", "N", "hydrate ventilate operate sedate intubate dilate infuse transfuse inject infect resect"),
       ("ABLE", "able", "V", "ADJ", "treat prevent detect measure cure avoid manage"),
       ("IVE", "ive", "V", "ADJ", "operate act sedate prevent protect restrict obstruct support infect disrupt"),
       ("MENT", "ment", "V", "N", "treat manage assess displace improve replace involve impair attach"),
       ("ITY", "ity", "ADJ", "N", "toxic active sensitive mobile sterile fertile"),
       ("NESS", "ness", "ADJ", "N", "ill weak sick dizzy drowsy tired numb stiff tender"),
       ("LY", "ly", "ADJ", "ADV", "severe mild slow rapid acute")]
PREFIX = "re un hyper hypo post pre anti intra nephro cardio peri sub".split()
PXLIST = [("re", "V", "admit"), ("re", "V", "assess"), ("re", "V", "examine"), ("re", "V", "operate"), ("re", "V", "insert"), ("un", "ADJ", "stable"), ("un", "ADJ", "conscious"),
          ("un", "ADJ", "intact"), ("hyper", "N", "tension"), ("hypo", "N", "tension"), ("intra", "ADJ", "venous"), ("intra", "ADJ", "muscular"), ("intra", "ADJ", "cranial"),
          ("anti", "N", "biotic"), ("anti", "N", "coagulant"), ("anti", "N", "pyretic"), ("anti", "N", "emetic"), ("nephro", "ADJ", "toxic"), ("cardio", "ADJ", "vascular"),
          ("sub", "ADJ", "cutaneous")]
INFL = [("s", "+PL", "noun plural"), ("es", "+PL", "after s,x,z,ch,sh"), ("ies", "+PL", "after consonant+y"), ("s", "+3SG", "verb 3rd singular"), ("es", "+3SG", "after sibilant"),
        ("ies", "+3SG", "after consonant+y"), ("ed", "+PAST", "past"), ("d", "+PAST", "after e (e-deletion)"), ("ied", "+PAST", "after consonant+y"), ("Ced", "+PAST", "consonant doubling"),
        ("ed", "+PASTP", "past participle"), ("d", "+PASTP", "after e"), ("ied", "+PASTP", "after consonant+y"), ("Ced", "+PASTP", "consonant doubling"),
        ("ing", "+PRESP", "present participle"), ("ing", "+PRESP", "e-deletion: make-ing"), ("Cing", "+PRESP", "consonant doubling"), ("er", "+COMP", "comparative"),
        ("est", "+SUP", "superlative"), ("ier", "+COMP", "y-to-i"), ("iest", "+SUP", "y-to-i"), ("'s", "+POSS", "possessive (not used in corpus)"), ("s'", "+POSS.PL", "plural possessive")]
W = lambda s: s.split()


# =====================================================================================
# 2. FINITE-STATE MACHINERY: orthographic rule transducers + lexicon FST (bidirectional)
# =====================================================================================
class Rule:
    """Rewrite rule  a -> b / left __ right  compiled into a deterministic windowed transducer.
    State = (last 3 consumed input symbols, look-ahead buffer).  Output may be delayed, never revised."""
    def __init__(s, name, a, b, left, right):
        s.name, s.a, s.b, s.left, s.right = name, a, b, left, right

    def step(s, st, ch, end=False):
        hist, buf = st
        buf += ch
        out = ""
        while buf:
            c, k = buf[0], len(s.right)
            if re.fullmatch(s.a, c) and re.search(s.left + r"\Z", hist):
                if len(buf) - 1 < k and not end:
                    break
                if len(buf) - 1 >= k and all(re.fullmatch(p, x) for p, x in zip(s.right, buf[1:1 + k])):
                    out += s.b(c) if callable(s.b) else s.b
                else:
                    out += c
            else:
                out += c
            hist, buf = (hist + c)[-3:], buf[1:]
        return (hist, buf), out

    def text(s):
        b = "doubled" if callable(s.b) else (s.b or "ε")
        return f"{s.a.replace(chr(92), '')} → {b} / {s.left.replace(chr(92), '') or '·'} __ {' '.join(x.replace(chr(92), '') for x in s.right) or '·'}"


C, V = "[bcdfghjklmnpqrstvwxyz]", "[aeiou]"
R_DBL = Rule("consonant-doubling", "[bdgmnprt]", lambda c: c + c, C + V, (r"\^", V))
R_EDEL = Rule("e-deletion", "e", "", "(?:" + C + "|u)", (r"[\^~]", V))
R_YIE = Rule("y-to-ie", "y", "ie", C, (r"\^", "s"))
R_YI = Rule("y-to-i", "y", "i", C, (r"[\^~]", r"[^i\^~|=#]"))
R_EINS = Rule("e-insertion", r"\^", "e", r"(?:s|x|z|ch|sh)", ("s", "#"))
R_CLEAN = Rule("boundary-cleanup", r"[\^~|=#]", "", "", ())
RULES = (R_DBL, R_EDEL, R_YIE, R_YI, R_EINS, R_CLEAN)


@functools.lru_cache(maxsize=None)
def feed(states, s, rules):
    states, out = list(states), ""
    for ch in s:
        pieces = ch
        for i, r in enumerate(rules):
            nxt = ""
            for p in pieces:
                states[i], o = r.step(states[i], p)
                nxt += o
            pieces = nxt
        out += pieces
    return tuple(states), out


def flush(states, rules):
    states, carry = list(states), ""
    for i, r in enumerate(rules):
        o = ""
        for p in carry:
            states[i], x = r.step(states[i], p)
            o += x
        states[i], x = r.step(states[i], "", True)
        carry = o + x
    return carry


class FST:
    """Lexicon transducer: lexical tape (stem letters + feature tags)  <->  intermediate tape (stem ^ suffix #)."""
    def __init__(s, lex, irr_v, irr_n, irr_a, use_exc=True, rules=RULES):
        s.arcs, s.n, s.final, s.cache, s.rules, s.trie = defaultdict(list), 0, set(), {}, rules, {}
        s.start, s.root, s.fin = s.new(), s.new(), s.new()
        s.final.add(s.fin)
        s.arc(s.start, "", "", s.root)
        for p in PREFIX:
            s.arc(s.start, p + "-", p + "=", s.root)
        for (stem, pos), info in lex.items():
            node = s.root
            for ch in stem:
                if (node, ch) not in s.trie:
                    s.trie[(node, ch)] = s.new()
                    s.arc(node, ch, ch, s.trie[(node, ch)])
                node = s.trie[(node, ch)]
            ib = "|" if (use_exc and stem in EXC) else "^"
            db = "|" if (use_exc and stem in EXC) else "~"
            if pos == "FUNC":
                s.arc(node, "+FUNC", "#", s.fin)
                continue
            s1 = s.new()
            s.arc(node, "+" + pos, "", s1)
            blocked = set()
            if pos == "V":
                blocked = set(irr_v.get(stem, {}))
                for t, o in (("BASE", "#"), ("3SG", ib + "s#"), ("PAST", ib + "ed#"), ("PASTP", ib + "ed#"), ("PRESP", ib + "ing#")):
                    if t not in blocked:
                        s.arc(s1, "+" + t, o, s.fin)
            elif pos == "N":
                s.arc(s1, "+SG", "#", s.fin)
                if stem not in irr_n:
                    s.arc(s1, "+PL", ib + "s#", s.fin)
            elif pos == "ADJ":
                s.arc(s1, "+BASE", "#", s.fin)
                if info["grad"]:
                    blocked = set(irr_a.get(stem, {}))
                    for t, o in (("COMP", ib + "er#"), ("SUP", ib + "est#")):
                        if t not in blocked:
                            s.arc(s1, "+" + t, o, s.fin)
            for tag, suf, fpos, tpos, stems in DER:
                if stem in stems.split() and pos in fpos.split("|"):
                    sd = s.new()
                    s.arc(s1, "+" + tag, db + suf, sd)
                    if tpos == "N":
                        sx = s.new(); s.arc(sd, "+N", "", sx); s.arc(sx, "+SG", "#", s.fin); s.arc(sx, "+PL", ib + "s#", s.fin)
                    elif tpos == "ADJ":
                        sx = s.new(); s.arc(sd, "+ADJ", "", sx); s.arc(sx, "+BASE", "#", s.fin)
                        if (stem, tag) == ("act", "IVE"):
                            sy = s.new(); s.arc(sx, "+ITY", db + "ity", sy); sz = s.new(); s.arc(sy, "+N", "", sz); s.arc(sz, "+SG", "#", s.fin); s.arc(sz, "+PL", ib + "s#", s.fin)
                    else:
                        s.arc(sd, "+ADV", "#", s.fin)
        # irregular chains: first letter emits the whole irregular form, the rest of the lexical tape emits nothing
        chains = [(st, "V", t, f) for st, d in irr_v.items() for t, fs in d.items() for f in fs if (st, "V") in lex]
        chains += [(st, "N", "PL", f) for st, f in irr_n.items() if (st, "N") in lex]
        chains += [(st, "ADJ", t, f) for st, d in irr_a.items() for t, fs in d.items() for f in fs if (st, "ADJ") in lex]
        for st, pos, tag, form in chains:
            q = s.root
            for i, ch in enumerate(st):
                nq = s.new(); s.arc(q, ch, form + "#" if i == 0 else "", nq); q = nq
            q2 = s.new(); s.arc(q, "+" + pos, "", q2); s.arc(q2, "+" + tag, "", s.fin)

    def new(s): s.n += 1; return s.n - 1
    def arc(s, a, i, o, b): s.arcs[a].append((i, o, b))
    def n_arcs(s): return sum(len(v) for v in s.arcs.values())

    def gen(s, lexical, trace=False):
        """generation direction: lexical string -> surface forms (follows the arcs whose input symbol matches the tape)"""
        toks = re.findall(r"[a-z]+-|\+[A-Z0-9.]+|[a-z]", lexical)
        res = []
        def dfs(q, i, st, em, path):
            if q in s.final and i == len(toks):
                res.append((em + flush(st, s.rules), path))
            for sym, o, b in s.arcs[q]:
                if sym == "" or (i < len(toks) and sym == toks[i]):
                    st2, out = feed(st, o, s.rules)
                    dfs(b, i + (sym != ""), st2, em + out, path + [(q, sym, o, b)])
        dfs(s.start, 0, tuple(("", "") for _ in s.rules), "", [])
        return res if trace else [r[0] for r in res]

    def parse(s, surface, trace=False):
        """analysis direction: the SAME arcs are explored; the generated output must stay a prefix of the surface"""
        key = (surface, trace)
        if key in s.cache: return s.cache[key]
        res = []
        def dfs(q, lex, st, em, path):
            if not surface.startswith(em): return
            if q in s.final:
                if em + flush(st, s.rules) == surface:
                    res.append(("".join(lex), path))
                return
            for sym, o, b in s.arcs[q]:
                st2, out = feed(st, o, s.rules)
                dfs(b, lex + [sym], st2, em + out, path + [(q, sym, o, b)])
        dfs(s.start, [], tuple(("", "") for _ in s.rules), "", [])
        s.cache[key] = res if trace else sorted({r[0] for r in res})
        return s.cache[key]

    def gen1(s, lexical):
        g = s.gen(lexical)
        return g[0] if g else None


def parallel(rules, s):
    """two-level (Koskenniemi-style) realisation: every rule looks at the ORIGINAL string simultaneously; first matching rule (most specific) decides."""
    out = []
    order = [R_DBL, R_EDEL, R_YIE, R_YI, R_EINS, R_CLEAN]
    for i, c in enumerate(s):
        piece = c
        for r in order:
            left, right = s[max(0, i - 3):i], s[i + 1:i + 1 + len(r.right)]
            if re.fullmatch(r.a, c) and re.search(r.left + r"\Z", left) and len(right) == len(r.right) and all(re.fullmatch(p, x) for p, x in zip(r.right, right)):
                piece = r.b(c) if callable(r.b) else r.b
                break
        out.append(piece)
    return "".join(out)


def build_lexicon():
    lex = {}
    def add(stems, pos, grad=False):
        for st in W(stems): lex[(st, pos)] = {"grad": grad}
    add(" ".join([SYMP, BODY, COND, PROC, ITEM, PERS, ABS]), "N")
    add(" ".join([VSURG, VMED, VOBS, VCARE, VSYM, VOTH, VIRR_L]), "V")
    add(AG, "ADJ", True); add(AN, "ADJ"); add("good bad", "ADJ", True)
    for tag, suf, fpos, tpos, stems in DER:
        for st in W(stems):
            if fpos == "CF": lex.setdefault((st, "CF"), {"grad": False})
            elif fpos == "N|CF" and (st, "N") not in lex: lex.setdefault((st, "CF"), {"grad": False})
            elif fpos in ("V", "ADJ") and (st, fpos) not in lex: raise SystemExit(f"missing stem {st} {fpos}")
    stems_open = len(lex)
    for w in W(FUNC): lex[(w, "FUNC")] = {"grad": False}
    return lex, stems_open


LEX, N_OPEN = build_lexicon()
FSTX = FST(LEX, IRR_V, IRR_N, IRR_A, use_exc=True)


def lexical_to_lemma(lexical):
    toks = re.findall(r"[a-z]+-|\+[A-Z0-9.]+|[a-z]", lexical)
    base = {"+PL": "+SG", "+SG": "+SG", "+3SG": "+BASE", "+PAST": "+BASE", "+PASTP": "+BASE", "+PRESP": "+BASE", "+COMP": "+BASE", "+SUP": "+BASE"}
    toks[-1] = base.get(toks[-1], toks[-1])
    return FSTX.gen1("".join(toks)) or lexical


def feats_of(lexical):
    return [t for t in re.findall(r"\+[A-Z0-9.]+", lexical)]


# =====================================================================================
# 3. SYNTHETIC CORPUS  (templated, documented; entities and lemmas are known by construction)
# =====================================================================================
DRUGS = W("Paracetamol Metformin Amlodipine Atorvastatin Ceftriaxone Amoxicillin Pantoprazole Ondansetron Enoxaparin Aspirin Clopidogrel Furosemide Ramipril Omeprazole "
          "Azithromycin Ibuprofen Diclofenac Tramadol Morphine Salbutamol Prednisolone Levothyroxine Gentamicin Meropenem Vancomycin Cefixime Telmisartan Glimepiride")
LABS = [("Hb", "g/dL", 7, 16, 1), ("Creat", "mg/dL", 0.5, 6, 1), ("Na", "mmol/L", 120, 150, 0), ("K", "mmol/L", 2.5, 6.5, 1), ("ALT", "U/L", 10, 200, 0), ("HbA1c", "%", 5, 12, 1), ("SpO2", "%", 85, 100, 0)]
ICDS = W("I10 E11.9 J18.9 K35.8 S72.0 N18.5 A09 C50.9 I21.9 K80.2 G40.9 J45.9 M17.1 N39.0 D64.9")
MON = "Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec".split()


def pick(seq): return seq[R.randrange(len(seq))]
def wchoice(pairs):
    x, acc = R.random() * sum(w for _, w in pairs), 0
    for v, w in pairs:
        acc += w
        if x <= acc: return v
    return pairs[-1][0]


def ent(kind):
    if kind == "PID": return ("PID", f"MRN-{R.randint(100000, 999999)}" if R.random() < .9 else f"UHID {R.randint(1000000, 9999999)}")
    if kind == "DATE":
        d, m, y = R.randint(1, 28), R.randint(1, 12), R.randint(2014, 2024)
        f = wchoice([(0, 55), (1, 15), (2, 18), (3, 12)])
        return ("DATE", [f"{d:02d}/{m:02d}/{y}", f"{d:02d}-{m:02d}-{y}", f"{d:02d} {MON[m - 1]} {y}", f"{y}-{m:02d}-{d:02d}"][f])
    if kind in ("DRUG", "DRUGN"):
        d, dose = pick(DRUGS), pick([5, 10, 20, 40, 50, 75, 100, 150, 250, 500, 625, 650, 1000])
        f = wchoice([(0, 80), (1, 8 if kind == "DRUGN" else 2), (2, 5), (3, 5)])
        if kind == "DRUGN" and f == 0 and R.random() < .45: f = 1
        return ("DRUG", [f"{d} {dose} mg", f"{d.lower()} {dose}mg", f"{d}-Clavulanate {dose} mg", f"{d} {pick([10, 20, 40])} IU"][f])
    if kind == "FREQ": return ("FREQ", wchoice([(pick(["OD", "BD", "TDS", "QID", "HS", "SOS", "PRN"]), 80), (pick(["twice daily", "once daily", "thrice a day"]), 14), ("1-0-1", 4), ("bid", 2)]))
    if kind == "LAB":
        n, u, lo, hi, dp = pick(LABS)
        v = round(R.uniform(lo, hi), dp) if dp else R.randint(int(lo), int(hi))
        return ("LAB", f"{v} {u}" if R.random() < .85 else f"{v}{u}")
    if kind == "BP": return ("BP", f"{R.randint(90, 180)}/{R.randint(55, 110)}" + (" mmHg" if R.random() < .7 else ""))
    if kind == "TEMP":
        t = round(R.uniform(36, 40), 1)
        return ("TEMP", wchoice([(f"{t} C", 40), (f"{t}°C", 30), (f"{round(t * 1.8 + 32, 1)} F", 20), (f"{t} deg C", 10)]))
    if kind == "ICD":
        c = pick(ICDS)
        return ("ICD", c.lower() if R.random() < .05 else c)
    if kind == "WARD":
        n, b = R.randint(1, 9), R.randint(1, 40)
        return ("WARD", wchoice([(f"Ward {n}{pick(['', 'A', 'B'])} Bed {b}", 88), (f"W{n}/B{b}", 12)]))
    if kind == "PHONE":
        p = f"{R.choice('6789')}{R.randint(100000000, 999999999)}"
        return ("PHONE", wchoice([(p, 60), ("+91-" + p, 25), (p[:5] + " " + p[5:], 15)]))
    if kind == "AGE": return ("AGE", wchoice([(f"{R.randint(1, 95)} yrs", 30), (f"{R.randint(1, 95)} y/o", 20), (f"{R.randint(1, 95)}-year-old", 25), (f"{R.randint(1, 95)} years old", 25)]))
    if kind == "DUR": return ("DUR", wchoice([(f"{R.randint(1, 14)} days", 40), (f"{R.randint(1, 8)} weeks", 25), (f"{R.randint(1, 12)} months", 15), (f"{R.randint(1, 20)} yrs", 10), (f"{R.randint(2, 72)} hrs", 10)]))
    raise KeyError(kind)


GROUPS = dict(SYMP=SYMP, BODY=BODY, COND=COND, PROC=PROC, ITEM=ITEM, PERS=PERS, ABS=ABS, VSURG=VSURG, VMED=VMED, VOBS=VOBS, VCARE=VCARE, VSYM=VSYM, VOTH=VOTH, VIRR=VIRR_L,
              AG=AG, AN=AN, SEX=SEX)
TEMPL = {
    "discharge": ["{E:PID} , {E:AGE} {A:SEX} , was {VL:admit:PASTP} on {E:DATE} to {E:WARD} with {N:SYMP} and {N:SYMP} for {E:DUR} .",
                  "on {N1:ABS} the {N1:BODY} was {A:AG} and the {N:BODY} were {A:AG:COMP} than the {N1:BODY} .",
                  "the {N1:ABS} was {D:ITIS} with {N:COND} ; {E:ICD} .",
                  "a {D:ECTOMY} was {V:VOTH:PASTP} on {E:DATE} and the {N1:ABS} was {A:AN} .",
                  "{LB:} {E:LAB} , {LB:} {E:LAB} , {E:BP} and {E:TEMP} were {V:VOBS:PASTP} .",
                  "{E:DRUG} {E:FREQ} was {V:VMED:PASTP} for {E:DUR} .",
                  "{N:SYMP} {V:VSYM:PAST} after the {N1:PROC} and the {N:ITEM} was {V:VCARE:PASTP} .",
                  "follow up after {E:DUR} ; {V:VMED:BASE} the {N:ITEM} and call {E:PHONE} .",
                  "advice : {V:VCARE:BASE} the {N1:BODY} and {V:VOBS:BASE} the {N:SYMP} , not {D:ION} .",
                  "the {N1:PERS} {V:VIRR:PAST} the {N1:ITEM} and the {X:} was {D:IVE} .",
                  "vitamin B12 was {V:VMED:PASTP} and a 16 F Foley {N1:ITEM} was {V:VSURG:PASTP} ."],
    "prescription": ["{E:PID} {E:AGE} {A:SEX} {E:DATE} .", "rx : {E:DRUG} {E:FREQ} for {E:DUR} ; {E:DRUG} {E:FREQ} for {E:DUR} .",
                     "{V:VMED:BASE} {E:DRUG} {E:FREQ} {V:VMED:PRESP} the {N1:ITEM} .", "do not {V:VOTH:BASE} the {N:ITEM} if {N:SYMP} {V:VSYM:3SG} .",
                     "the {N1:PERS} was {V:VMED:PASTP} {N:ITEM} and {N1:ITEM} for {N1:COND} ; {E:ICD} .", "{E:DRUG} {E:FREQ} ; {E:DRUG} {E:FREQ} ; the {N1:ITEM} was {V:VMED:PASTP} by the {N1:PERS} .",
                     "{LB:} {E:LAB} , {E:BP} .", "contact the {N1:PERS} on {E:PHONE} , {E:WARD} .", "Administer 2 mg of the {N1:ITEM} , lab no 9123456780 .", "the {A:AN} {N:SYMP} {V:VSYM:PRESP} , the {X:} was {A:AG:SUP} ."],
    "operation": ["{E:PID} {E:DATE} : {D:ECTOMY} ; {D:OTOMY} .", "the {N1:PERS} {V:VSURG:PAST} the {N1:BODY} and {V:VSURG:PAST} the {N:ITEM} .", "{D:ION} was {A:AG} ; the {N1:BODY} was {A:AN} .",
                  "{E:BP} , {E:TEMP} , {LB:} {E:LAB} during the {N1:PROC} .", "the {N:ITEM} was {V:VSURG:PASTP} and the {N1:BODY} {V:VSYM:PAST} ; the {N1:ABS} was {D:IVE} .",
                  "{V:VOBS:PRESP} the {N:SYMP} , the {N1:PERS} {V:VIRR:PAST} the {N1:BODY} .", "{E:ICD} , {D:OLOGY} , {D:OSIS} and {D:AEMIA} were {V:VOBS:PASTP} .", "a {X:} {N1:PROC} was {V:VOTH:PASTP} on {E:DATE} .",
                  "the {A:AN} {N1:COND} {V:VSYM:PAST} and the {N1:BODY} was {D:MENT} ."],
    "nursing": ["{E:DATE} {E:WARD} : the {N1:PERS} {V:VSYM:PAST} {N:SYMP} .", "bp {E:BP} , temp {E:TEMP} , {LB:} {E:LAB} .", "pt {V:VSYM:PAST} {N:SYMP} for {E:DUR} , {V:VCARE:PASTP} by the {N1:PERS} .",
                "{E:DRUGN} {E:FREQ} {V:VMED:PASTP} ; {N1:SYMP} {V:VSYM:PAST} .", "the {N:ITEM} {V:VCARE:PASTP} , the {N1:BODY} was {A:AG:COMP} .",
                "the {N1:PERS} {V:VIRR:PAST} {N1:ABS} ; the {N1:PERS} {V:VOBS:PAST} {A:AN} {N:SYMP} .", "{E:PID} {E:AGE} , {E:WARD} , {A:AN} {N:SYMP} for {E:DUR} .", "vision OD 6/9 , OS 6/6 ; {V:VMED:PASTP} .",
                "{D:ABLE} {N:SYMP} , {D:NESS} {V:VSYM:PRESP} , {D:LY} {A:AG} ."]}
TEMPL["discharge"] += ["the {N:ITEM} {V:VOBS:PASTP} , {N:SYMP} {V:VSYM:PAST} , the {N1:ABS} was {A:AN} and {A:AG:COMP} ."]
LOG = []  # per-token gold: (surface, lemma, feats, category, lang)


class Rec:
    def __init__(s, rtype): s.rtype, s.t, s.ents, s.lem = rtype, [], [], []
    def lit(s, w): s.t.append(w)
    def ent(s, kind):
        k, txt = ent(kind); s.ents.append((k, txt)); s.t.append(txt)
    def word(s, lexical, typo=True):
        surf = FSTX.gen1(lexical)
        assert surf, lexical
        lem = lexical_to_lemma(lexical)
        m = re.match(r"(?:[a-z]+-)?([a-z]+)\+(\w+)", lexical); last = lexical.split("+")[-1]
        irr = m and ((m[1] in IRR_V and last in IRR_V[m[1]]) or (m[1] in IRR_N and last == "PL") or (m[1] in IRR_A and last in IRR_A[m[1]]))
        cat = "irr" if irr else "der" if (re.search(r"\+(ITIS|ECTOMY|OTOMY|OLOGY|OSIS|AEMIA|IC|AL|ATION|ION|ABLE|IVE|MENT|ITY|NESS|LY)", lexical) or "-" in lexical) else "reg"
        feats, typo_flag = feats_of(lexical), False
        if typo and len(surf) >= 6 and R.random() < .02:
            i = R.randrange(1, len(surf) - 2); surf = surf[:i] + surf[i + 1] + surf[i] + surf[i + 2:]; typo_flag = True
        s.t.append(surf); s.lem.append((surf, lem, "".join(feats) + ("+TYPO" if typo_flag else ""), "typo" if typo_flag else cat, "en"))
        return surf
    def text(s): return re.sub(r" ([.,;:])", r"\1", " ".join(s.t))


def lex_noun(stem, pl): return f"{stem}+N+{'PL' if pl else 'SG'}"
def derived_tokens(tag):
    t, suf, fpos, tpos, stems = next(d for d in DER if d[0] == tag)
    st = pick(W(stems))
    pos = fpos if "|" not in fpos else ("N" if (st, "N") in LEX else "CF")
    base = f"{st}+{pos}+{tag}"
    if tpos == "N": return base + f"+N+{'PL' if (R.random() < .3 and tag in ('ECTOMY', 'OTOMY', 'OLOGY', 'ION', 'ATION', 'MENT', 'AEMIA')) else 'SG'}"
    if tpos == "ADJ": return base + "+ADJ+BASE"
    return base + "+ADV"


def prefixed():
    if R.random() < .25:
        p = pick(["pre", "post", "peri"]); return f"{p}-" + "operate+V+IVE+ADJ+BASE"
    if R.random() < .15: return pick(["hypo", "hyper"]) + "-glyc+CF+AEMIA+N+SG"
    p, pos, st = pick(PXLIST)
    tag = {"V": pick(["PAST", "PASTP", "PRESP"]), "N": pick(["SG", "PL"]), "ADJ": "BASE"}[pos]
    return f"{p}-{st}+{pos}+{tag}"


def fill(rec, template):
    for tok in template.split():
        m = re.fullmatch(r"\{(\w+):([^}]*)\}", tok)
        if not m: rec.lit(tok); continue
        k, a = m.group(1), m.group(2).split(":")
        if k == "E": rec.ent(a[0])
        elif k == "LB": rec.lit(pick(["Hb", "Creat", "Na", "K", "ALT", "HbA1c", "SpO2"]))
        elif k in ("N", "N1", "NP"):
            st = pick(W(GROUPS[a[0]])); pl = (k == "NP") or (k == "N" and R.random() < .35 and st not in MASS)
            rec.word(lex_noun(st, pl))
        elif k == "V": rec.word(f"{pick(W(GROUPS[a[0]]))}+V+{a[1]}")
        elif k == "VL": rec.word(f"{a[0]}+V+{a[1]}", typo=False)
        elif k == "A": rec.word(f"{pick(W(GROUPS[a[0]]))}+ADJ+{a[1] if len(a) > 1 else 'BASE'}")
        elif k == "D": rec.word(derived_tokens(a[0]))
        elif k == "X": rec.word(prefixed())
    return rec


def make_record(rtype, rng_templates=None):
    rec = Rec(rtype)
    ts = list(TEMPL[rtype])
    first = ts[0] if rtype in ("discharge", "prescription") else None
    chosen = ([first] if first else []) + R.sample(ts[1:] if first else ts, R.randint(6, 8) if rtype != "prescription" else R.randint(5, 7))
    for t in chosen: fill(rec, t)
    return rec


def build_corpus(n_per=130):
    recs = []
    for rt in ("discharge", "prescription", "operation", "nursing"):
        for _ in range(n_per): recs.append(make_record(rt))
    return recs


# =====================================================================================
# 4. REGEX PATTERNS (Task 2), TOKENIZER + NORMALISER (Task 3)
# =====================================================================================
MONS = "|".join(MON)
PAT = {"PID": r"(?<!\w)MRN-\d{6}(?!\w)",
       "DATE": rf"(?<![\w/-])(?:\d{{1,2}}[/-]\d{{1,2}}[/-]\d{{2,4}}|\d{{4}}-\d{{2}}-\d{{2}}|\d{{1,2}} (?:{MONS}) \d{{4}})(?![\w/-])",
       "DRUG": r"(?<!\w)[A-Z][a-z]{3,}\s\d+(?:\.\d+)?\s?(?:mg|mcg|g|ml|IU)(?!\w)",
       "FREQ": r"(?<!\w)(?:OD|BD|TDS|QID|HS|SOS|PRN|(?:once|twice|thrice) (?:a |per )?(?:daily|day))(?!\w)",
       "LAB": r"(?<![\w.])\d+(?:\.\d+)?\s?(?:mg/dL|g/dL|mmol/L|mEq/L|U/L|%)(?!\w)",
       "BP": r"(?<![\w/])\d{2,3}/\d{2,3}(?:\s?mmHg)?(?![\w/])",
       "TEMP": r"(?<![\w.])\d{2,3}(?:\.\d)?\s?°?\s?[CF](?!\w)",
       "ICD": r"(?<![\w.])[A-TV-Z]\d{2}(?:\.\d{1,2})?(?!\w|\.\d)",
       "WARD": r"(?<!\w)Ward \d{1,2}[A-Z]? Bed \d{1,3}(?!\w)",
       "PHONE": r"(?<![\w+])(?:\+91[- ]?)?[6-9]\d{9}(?!\w)",
       "AGE": r"(?<![\w.])\d{1,3}\s?(?:yo|y/o|yrs|years old|-year-old)(?!\w)",
       "DUR": r"(?<![\w.])\d+\s?(?:days?|weeks?|months?|hrs?|hours?|yrs?|years?)(?!\w)"}
CPAT = {k: re.compile(v) for k, v in PAT.items()}


def find_entities(text):
    out = []
    for k, rx in CPAT.items():
        out += [(k, m.group(0)) for m in rx.finditer(text)]
    return out


def prf(found, gold):
    f, g = Counter(found), Counter(gold)
    tp = sum((f & g).values()); return tp, sum(f.values()) - tp, sum(g.values()) - tp


TOKP = [("DATE", PAT["DATE"].replace("(?<![\\w/-])", "").replace("(?![\\w/-])", "")), ("ABBR", r"(?:[A-Za-z]\.){2,}|(?:Dr|Mr|Mrs|Ms|Pt)\."), ("UNIT", r"(?:mg|g|mmol|mEq|U|mcg)/(?:dL|L|mL|kg|h)"),
        ("BP", r"\d{2,3}/\d{2,3}"), ("NUM", r"\d+(?:\.\d+)?"), ("WORD", r"[A-Za-z][A-Za-z0-9]*(?:[-'][A-Za-z0-9]+)*"), ("PUNCT", r"[^\sA-Za-z0-9]")]
TOKC = [(n, re.compile(p)) for n, p in TOKP]


def tokenize(text):
    """maximal munch: at each position take the LONGEST match over all token classes (ties -> earlier class)"""
    i, out = 0, []
    while i < len(text):
        if text[i].isspace(): i += 1; continue
        best = max(((len(m.group(0)), -k, n, m.group(0)) for k, (n, rx) in enumerate(TOKC) for m in [rx.match(text, i)] if m), default=None)
        out.append((best[2], best[3])); i += best[0]
    return out


ABBR = {"pt": "patient", "c/o": "complains of", "h/o": "history of", "s/p": "status post", "adm": "admitted", "dx": "diagnosis", "rx": "prescription", "tx": "treatment", "hx": "history",
        "sx": "symptoms", "fx": "fracture", "bp": "blood pressure", "hr": "heart rate", "rr": "respiratory rate", "temp": "temperature", "spo2": "oxygen saturation", "ecg": "electrocardiogram",
        "bd": "twice daily", "od": "once daily", "tds": "three times daily", "qid": "four times daily", "prn": "as needed", "sos": "if needed", "hs": "at bedtime", "iv": "intravenous",
        "im": "intramuscular", "po": "by mouth", "npo": "nil by mouth", "tab": "tablet", "cap": "capsule", "inj": "injection", "sob": "shortness of breath", "cp": "chest pain",
        "dm": "diabetes mellitus", "htn": "hypertension", "ckd": "chronic kidney disease", "uti": "urinary tract infection", "hb": "haemoglobin"}


def iso_date(m):
    s = m.group(0)
    a = re.fullmatch(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})", s)
    if a:
        d, mo, y = int(a[1]), int(a[2]), int(a[3]); y += 2000 if y < 100 else 0; return f"{y}-{mo:02d}-{d:02d}"
    a = re.fullmatch(r"(\d{1,2}) (\w{3}) (\d{4})", s)
    return f"{a[3]}-{MON.index(a[2]) + 1:02d}-{int(a[1]):02d}" if a else s


def normalise(text):
    t = re.sub(PAT["DATE"], iso_date, text)                                        # dates -> ISO (assumes dd/mm)
    t = re.sub(r"(\d)\s?(mg|mcg|g|ml|IU|kg)\b", lambda m: f"{m[1]} {m[2].lower()}", t, flags=re.I)   # unit canonical form
    t = re.sub(r"(\d)\s?°\s?([CF])", r"\1 \2", t)
    t = t.lower()                                                                   # case folding
    for k in sorted(ABBR, key=len, reverse=True):
        t = re.sub(rf"(?<![\w/]){re.escape(k)}(?![\w/])", ABBR[k], t)               # abbreviation expansion
    return re.sub(r"\s+", " ", t).strip()


# =====================================================================================
# 5. AUTOMATA (Task 4): DFA, NFA with epsilon, Thompson, subset construction, minimisation, state elimination
# =====================================================================================
class DFA:
    def __init__(s, edges, finals, cl, q0=0, labels=None):
        s.d, s.F, s.cl, s.q0 = dict(edges), set(finals), cl, q0
        s.Q = sorted({q0} | {q for (q, _) in s.d} | set(s.d.values())); s.sigma = sorted({c for (_, c) in s.d}); s.labels = labels or {}
    def accepts(s, w):
        q = s.q0
        for ch in w:
            q = s.d.get((q, s.cl(ch)))
            if q is None: return False
        return q in s.F
    def table(s):
        return [[q] + [s.d.get((q, c), "-") for c in s.sigma] + ["F" if q in s.F else ""] for q in s.Q]


def dfa_chain_specs():
    dg = lambda c: "d" if c.isdigit() else c
    mrn = DFA({(0, "M"): 1, (1, "R"): 2, (2, "N"): 3, (3, "-"): 4, **{(i, "d"): i + 1 for i in range(4, 10)}}, {10}, lambda c: c if c in "MRN-" else dg(c))
    icd = DFA({(0, "L"): 1, (1, "d"): 2, (2, "d"): 3, (3, "."): 4, (4, "d"): 5, (5, "d"): 6}, {3, 5, 6},
              lambda c: "L" if (c.isupper() and c != "U") else ("d" if c.isdigit() else c))
    bp = DFA({(0, "d"): 1, (1, "d"): 2, (2, "d"): 3, (2, "/"): 4, (3, "/"): 4, (4, "d"): 5, (5, "d"): 6, (6, "d"): 7}, {6, 7}, lambda c: dg(c))
    date = DFA({(0, "d"): 1, (1, "d"): 2, (1, "/"): 3, (2, "/"): 3, (3, "d"): 4, (4, "d"): 5, (4, "/"): 6, (5, "/"): 6, (6, "d"): 7, (7, "d"): 8, (8, "d"): 9, (9, "d"): 10}, {10}, lambda c: dg(c))
    return {"MRN identifier": mrn, "ICD-10 code": icd, "Blood pressure": bp, "Date dd/mm/yyyy": date}


class NFA:
    """edges: (q, label) -> set(q'); label '' = epsilon; label matched by `match`"""
    def __init__(s, match, q0=0, finals=()):
        s.t, s.match, s.q0, s.F, s.n = defaultdict(set), match, q0, set(finals), 0
    def new(s): s.n += 1; return s.n - 1
    def add(s, a, l, b): s.t[(a, l)].add(b)
    def closure(s, S):
        S, st = set(S), list(S)
        while st:
            q = st.pop()
            for b in s.t.get((q, ""), ()):
                if b not in S: S.add(b); st.append(b)
        return frozenset(S)
    def move(s, S, ch):
        return s.closure({b for (q, l), bs in s.t.items() if q in S and l and s.match(l, ch) for b in bs})
    def trace(s, w):
        S, rows = s.closure({s.q0}), []
        rows.append(("start", sorted(S)))
        for ch in w:
            S = s.move(S, ch); rows.append((ch, sorted(S)))
        return rows, bool(S & s.F)


def thompson(rx):
    nfa = NFA(lambda l, ch: re.fullmatch(l, ch) is not None); pos = [0]
    peek = lambda: rx[pos[0]] if pos[0] < len(rx) else None
    def eat(): pos[0] += 1; return rx[pos[0] - 1]
    def atom():
        c = eat()
        if c == "(":
            f = alt(); eat(); return f
        lab = c + eat() if c == "\\" else ("[" + "".join(iter(eat, "]")) + "]" if c == "[" else re.escape(c))
        a, b = nfa.new(), nfa.new(); nfa.add(a, lab, b); return a, b
    def rep():
        a, b = atom()
        while peek() in ("*", "+", "?"):
            op, s, e = eat(), nfa.new(), nfa.new()
            nfa.add(s, "", a); nfa.add(b, "", e)
            if op in "*?": nfa.add(s, "", e)
            if op in "*+": nfa.add(b, "", a)
            a, b = s, e
        return a, b
    def cat():
        fs = []
        while peek() not in (None, "|", ")"): fs.append(rep())
        a, b = fs[0]
        for c, d in fs[1:]: nfa.add(b, "", c); b = d
        return a, b
    def alt():
        fs = [cat()]
        while peek() == "|":
            eat(); fs.append(cat())
        if len(fs) == 1: return fs[0]
        s, e = nfa.new(), nfa.new()
        for a, b in fs: nfa.add(s, "", a); nfa.add(b, "", e)
        return s, e
    a, b = alt(); nfa.q0, nfa.F = a, {b}
    return nfa


UNIV = string.digits + string.ascii_letters + " .-/+%:,()°"


def subset(nfa, reps):
    """subset construction over the alphabet classes `reps` (name -> representative char). Returns DFA + construction table."""
    start = nfa.closure({nfa.q0}); names, order, delta, rows = {start: 0}, [start], {}, []
    i = 0
    while i < len(order):
        S = order[i]; row = {}
        for cname, rep in reps.items():
            T = nfa.move(S, rep)
            if T:
                if T not in names: names[T] = len(order); order.append(T)
                delta[(names[S], cname)] = names[T]; row[cname] = names[T]
        rows.append((names[S], sorted(S), row, bool(S & nfa.F))); i += 1
    return delta, {names[S] for S in order if S & nfa.F}, rows


def minimise(delta, finals, n, sigma):
    """Moore partition refinement on the complete DFA (a dead state is added implicitly)."""
    dead = n; Q = list(range(n)) + [dead]
    d = lambda q, c: delta.get((q, c), dead)
    part = {q: int(q in finals) for q in Q}
    while True:
        sig = {q: (part[q],) + tuple(part[d(q, c)] for c in sigma) for q in Q}
        ids = {v: i for i, v in enumerate(sorted(set(sig.values())))}
        new = {q: ids[sig[q]] for q in Q}
        if len(set(new.values())) == len(set(part.values())): break
        part = new
    return part, len(set(part[q] for q in range(n))), dead


def dfa_to_regex(Q, delta, q0, F, lab):
    live = {q for q in Q if any(q == f or True for f in F)}
    # keep only states that can reach a final state
    rev = defaultdict(set)
    for (q, c), r in delta.items(): rev[r].add(q)
    co, st = set(F), list(F)
    while st:
        x = st.pop()
        for p in rev[x]:
            if p not in co: co.add(p); st.append(p)
    union = lambda a, b: b if a is None else a if b is None else f"(?:{a}|{b})"
    cat = lambda a, b: None if a is None or b is None else a + b
    star = lambda a: "" if a in (None, "") else f"(?:{a})*"
    Rm = defaultdict(lambda: None)
    for (q, c), r in delta.items():
        if q in co and r in co: Rm[(q, r)] = union(Rm[(q, r)], lab[c])
    Rm[("s", q0)] = ""
    for f in F: Rm[(f, "a")] = union(Rm[(f, "a")], "") if Rm[(f, "a")] is not None else ""
    nodes = [q for q in Q if q in co]
    for k in nodes:
        loop = star(Rm[(k, k)])
        ins = [i for i in ["s"] + nodes if i != k and Rm[(i, k)] is not None]
        outs = [j for j in nodes + ["a"] if j != k and Rm[(k, j)] is not None]
        for i in ins:
            for j in outs: Rm[(i, j)] = union(Rm[(i, j)], cat(cat(Rm[(i, k)], loop), Rm[(k, j)]))
        for key in [x for x in list(Rm) if k in x]: del Rm[key]
    return Rm[("s", "a")]


def temp_nfa():
    cl = lambda c: "D" if c.isdigit() else "." if c == "." else " " if c == " " else "U" if c in "CF" else "x"
    n = NFA(lambda l, ch: cl(ch) == l, 0, {6})
    for a, l, b in [(0, "D", 1), (1, "D", 2), (2, ".", 3), (3, "D", 4), (2, "", 4), (4, " ", 5), (4, "", 5), (5, "U", 6)]: n.add(a, l, b)
    return n, cl


# =====================================================================================
# 6. HELPERS: weighted edit distance, language ID, OCR noise (Task 12)
# =====================================================================================
CONF = {("0", "o"), ("1", "l"), ("l", "i"), ("1", "i"), ("5", "s"), ("8", "b"), ("c", "e"), ("u", "v")}


def wed(a, b):
    n, m = len(a), len(b); D = [[0.0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): D[i][0] = i
    for j in range(m + 1): D[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            c = 0 if a[i - 1] == b[j - 1] else (.3 if (a[i - 1], b[j - 1]) in CONF or (b[j - 1], a[i - 1]) in CONF else 1)
            D[i][j] = min(D[i - 1][j] + 1, D[i][j - 1] + 1, D[i - 1][j - 1] + c)
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]: D[i][j] = min(D[i][j], D[i - 2][j - 2] + .7)
            if i > 1 and a[i - 2:i] == "rn" and b[j - 1] == "m": D[i][j] = min(D[i][j], D[i - 2][j - 1] + .3)
            if j > 1 and b[j - 2:j] == "rn" and a[i - 1] == "m": D[i][j] = min(D[i][j], D[i - 1][j - 2] + .3)
    return D[n][m]


TAMIL = {"kaichal": "fever", "vali": "pain", "irumal": "cough", "thalaivali": "headache", "vayiru": "abdomen", "vaanthi": "nausea", "veekkam": "swelling", "thadippu": "rash",
         "mathirai": "tablet", "nenju": "chest", "kaal": "foot", "kuduthen": "give", "saapten": "eat", "thoongunen": "sleep", "vizhundhen": "fall", "rattham": "blood"}
TAFUNC = W("ku la irundhuchu romba konjam illa seri naal innum")
TAVAR = {"kaichal": ["kaichal", "kaicchal", "kaichel"], "vali": ["vali", "valli", "vaali"], "irumal": ["irumal", "irumbal", "irumaal"], "thalaivali": ["thalaivali", "thalaivaali", "talaivali"],
         "mathirai": ["mathirai", "mathrai", "maathirai"], "kuduthen": ["kuduthen", "koduthen", "kuduthaen"], "irundhuchu": ["irundhuchu", "irunthuchu", "irundhuchi"],
         "vayiru": ["vayiru", "vayiRu", "vairu"], "nenju": ["nenju", "nenjhu", "nenju"]}


def ngrams(w): w = f"^{w}$"; return [w[i:i + 3] for i in range(len(w) - 2)]


class LangID:
    def __init__(s, en_words, ta_words):
        s.m = {}
        for lang, ws in (("en", en_words), ("ta", ta_words)):
            c = Counter(g for w in ws for g in ngrams(w)); s.m[lang] = (c, sum(c.values()))
        s.V = len({g for l in s.m.values() for g in l[0]})
    def predict(s, w):
        sc = {l: sum(math.log((c.get(g, 0) + 1) / (t + s.V)) for g in ngrams(w)) for l, (c, t) in s.m.items()}
        return max(sc, key=sc.get)


CONFM = {"o": "0", "l": "1", "m": "rn", "e": "c", "i": "l", "a": "o", "0": "O", "1": "l", "5": "S", "S": "5", "8": "B"}


def ocr_noise(text, pc=.035, pd=.08, ph=.07):
    out = []
    for w in text.split(" "):
        if len(w) >= 8 and w.isalpha() and R.random() < ph:
            k = R.randrange(3, len(w) - 2); w = w[:k] + "-\n" + w[k:]
        o = ""
        for ch in w:
            p = pd if ch.isdigit() else pc if ch.isalpha() else 0
            o += CONFM[ch] if ch in CONFM and R.random() < p else ch
        out.append(o)
    return " ".join(out)


def repair_ocr(text):
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", text).replace("\n", " ")
    def fix(m):
        s = m.group(0); digits = sum(c.isdigit() for c in s)
        return s.translate(str.maketrans("OolISB", "001158")) if digits and digits >= len(s) / 2 else s
    return re.sub(r"[0-9OolISB][0-9OolISB.,/\-]*[0-9OolISB]", fix, t)


CM_T = ["{N1:PERS} ku {N:SYMP} {TA:be} , {E:DRUGN} {E:FREQ} {TA:give} .", "{E:PID} {E:AGE} , {TA:body} {TA:pain} {TA:be} , {E:DRUGN} {E:FREQ} .",
        "{E:DATE} {TA:fever} {TA:be} romba , {E:TEMP} , {N1:ITEM} {TA:give} .", "{N1:PERS} {TA:cough} la {N:SYMP} , {E:BP} , {E:DRUGN} {E:FREQ} {TA:give} .",
        "{N:SYMP} konjam {TA:be} , {TA:body} {TA:pain} illa , {E:WARD} ."]


def code_mixed_record():
    rec = Rec("codemixed"); rec.lang = []
    for t in R.sample(CM_T, 4) + R.sample(TEMPL["nursing"], 3):
        for tok in t.split():
            m = re.fullmatch(r"\{TA:(\w+)\}", tok)
            if m:
                k = m.group(1)
                if k == "be": w = pick(TAVAR["irundhuchu"]); rec.t.append(w); rec.lem.append((w, "be", "+TA", "ta", "ta"))
                elif k == "give": w = pick(TAVAR["kuduthen"]); rec.t.append(w); rec.lem.append((w, "give", "+TA", "ta", "ta"))
                elif k == "body": w = pick(["nenju", "kaal", "vayiru"]); rec.t.append(w); rec.lem.append((w, {"nenju": "chest", "kaal": "foot", "vayiru": "abdomen"}[w], "+TA", "ta", "ta"))
                else:
                    key = {"pain": "vali", "fever": "kaichal", "cough": "irumal"}[k]; w = pick(TAVAR[key]); rec.t.append(w); rec.lem.append((w, TAMIL[key], "+TA", "ta", "ta"))
            else: fill_one(rec, tok)
    return rec


def fill_one(rec, tok):
    sub = Rec("tmp")
    fill(sub, tok)
    rec.t += sub.t; rec.ents += sub.ents; rec.lem += sub.lem
    if tok in ("ku", "la", "romba", "konjam", "illa"): rec.lem.append((tok, tok, "+TAFUNC", "taf", "ta"))


# =====================================================================================
# 7. EXPERIMENTS
# =====================================================================================
def words_of(text): return [w for n, w in tokenize(text) if n == "WORD"]
def lem_terms(fst, w):
    a = fst.parse(w.lower())
    return {lexical_to_lemma(x) for x in a} if a else {w.lower()}


def paradigm_forms():
    forms = {}
    for (st, pos), info in LEX.items():
        tags = {"N": ["SG", "PL"], "V": ["BASE", "3SG", "PAST", "PASTP", "PRESP"], "ADJ": ["BASE"] + (["COMP", "SUP"] if info["grad"] else []), "CF": [], "FUNC": []}[pos]
        for t in tags:
            for f in FSTX.gen(f"{st}+{pos}+{t}"): forms[f] = st
    for tag, suf, fpos, tpos, stems in DER:
        for st in W(stems):
            pos = fpos if "|" not in fpos else ("N" if (st, "N") in LEX else "CF")
            tails = ["+N+SG", "+N+PL"] if tpos == "N" else ["+ADJ+BASE"] if tpos == "ADJ" else ["+ADV"]
            for tl in tails:
                for f in FSTX.gen(f"{st}+{pos}+{tag}{tl}"): forms[f] = st
    return forms


def build_re_dfa(rx):
    nfa = thompson(rx)
    labels = sorted({l for (_, l) in nfa.t if l})
    classes = defaultdict(list)
    for ch in UNIV: classes[tuple(bool(re.fullmatch(l, ch)) for l in labels)].append(ch)
    classes = {sg: cs for sg, cs in classes.items() if any(sg)}
    names = {sg: f"c{i}" for i, sg in enumerate(classes)}
    reps = {names[sg]: cs[0] for sg, cs in classes.items()}
    members = {names[sg]: "".join(cs) for sg, cs in classes.items()}
    delta, finals, rows = subset(nfa, reps)
    n = len(rows); part, nmin, dead = minimise(delta, finals, n, list(reps))
    cmap = {ch: names[sg] for sg, cs in classes.items() for ch in cs}
    dfa = DFA(delta, finals, lambda c: cmap.get(c, "?"))
    return dict(nfa=nfa, nfa_states=nfa.n, dfa=dfa, dfa_states=n, min_states=nmin, rows=rows, reps=reps, members=members, part=part, labels=labels)


def eq_test(accept, rx, chars, n=3000):
    bad = 0
    for _ in range(n):
        w = "".join(R.choice(chars) for _ in range(R.randint(0, 14)))
        bad += accept(w) != (re.fullmatch(rx, w) is not None)
    return bad, n


def main():
    t0 = time.time()
    # ---------------- corpus ----------------
    recs = build_corpus(130)
    texts = [r.text() for r in recs]
    tok_total = sum(len([1 for n, _ in tokenize(t) if n != "PUNCT"]) for t in texts)
    types = Counter(w.lower() for t in texts for w in words_of(t))
    ent_counts = Counter(k for r in recs for k, _ in r.ents)
    OUT["corpus"] = dict(records=len(recs), by_type=dict(Counter(r.rtype for r in recs)), word_tokens=tok_total, alpha_types=len(types), entity_counts=dict(ent_counts),
                         typo_rate=round(sum(1 for r in recs for l in r.lem if l[3] == "typo") / sum(len(r.lem) for r in recs), 4),
                         samples={rt: next(t for r, t in zip(recs, texts) if r.rtype == rt) for rt in ("discharge", "prescription", "operation", "nursing")})
    print(f"corpus: {len(recs)} records, {tok_total} word tokens, {len(types)} alphabetic types")
    # ---------------- lexicon facts ----------------
    stems = {s for (s, p) in LEX if p != "FUNC"}
    bypos = Counter(p for (s, p) in LEX if p != "FUNC")
    OUT["lexicon"] = dict(distinct_stems=len(stems), entries=N_OPEN, by_pos=dict(bypos), func=len(W(FUNC)), infl_affixes=len(INFL), infl_table=INFL,
                          deriv_suffixes=[(t, "-" + s, f, o, len(W(st))) for t, s, f, o, st in DER], prefixes=PREFIX,
                          irr_v=len(IRR_V), irr_n=len(IRR_N), irr_a=len(IRR_A), irr_forms=sum(len(f) for d in IRR_V.values() for f in d.values()) + len(IRR_N) + 4,
                          fst_states=FSTX.n, fst_arcs=FSTX.n_arcs(), exc=sorted(EXC))
    print("lexicon:", OUT["lexicon"]["distinct_stems"], "stems,", FSTX.n, "FST states,", FSTX.n_arcs(), "arcs")

    # ---------------- Task 2 patterns ----------------
    res, fpex, fnex = {}, defaultdict(Counter), defaultdict(Counter)
    for k in PAT:
        TP = FP = FN = 0
        for r, t in zip(recs, texts):
            f = [x for kk, x in find_entities(t) if kk == k]; g = [x for kk, x in r.ents if kk == k]
            a, b, c = prf(f, g); TP += a; FP += b; FN += c
            for x in (Counter(f) - Counter(g)): fpex[k][x] += 1
            for x in (Counter(g) - Counter(f)): fnex[k][x] += 1
        res[k] = dict(tp=TP, fp=FP, fn=FN, p=round(TP / max(1, TP + FP), 3), r=round(TP / max(1, TP + FN), 3), pattern=PAT[k],
                      fp_ex=[x for x, _ in fpex[k].most_common(4)], fn_ex=[x for x, _ in fnex[k].most_common(4)])
    allTP, allFP, allFN = (sum(v[x] for v in res.values()) for x in ("tp", "fp", "fn"))
    OUT["task2"] = dict(patterns=res, micro_p=round(allTP / (allTP + allFP), 3), micro_r=round(allTP / (allTP + allFN), 3))
    s = "Tab Paracetamol 500 mg TDS and Tab Ibuprofen 400 mg BD"
    OUT["task2"]["greedy"] = dict(text=s, greedy=re.search(r"Tab .*mg", s).group(0), lazy=re.search(r"Tab .*?mg", s).group(0))
    t = ["MRN-123456", "MRN", "ward", "bed12", "ward12", "bed"]
    OUT["task2"]["precedence"] = dict(wrong=r"^ward|bed\d+$", right=r"^(?:ward|bed)\d+$", tests={x: (bool(re.search(r"^ward|bed\d+$", x)), bool(re.fullmatch(r"(?:ward|bed)\d+", x))) for x in ["ward", "bed12", "ward12", "wardrobe", "bed", "xbed7"]})
    print("task2 micro P/R", OUT["task2"]["micro_p"], OUT["task2"]["micro_r"])

    # ---------------- Task 3 tokenizer / normaliser ----------------
    ex = ["Hb 11.2g/dL post-operative, b.d. since 03/04/2024.", "BP 120/80mmHg; T 98.6F (afebrile).", "Dr. Rao: Amoxicillin-Clavulanate 625mg TDS x5days"]
    OUT["task3"] = dict(examples=[dict(text=e, split=e.split(), munch=[w for n, w in tokenize(e)]) for e in ex])
    rec0 = recs[260]
    OUT["task3"]["record_before"], OUT["task3"]["record_after"] = rec0.text(), normalise(rec0.text())
    OUT["task3"]["n_abbr"] = len(ABBR)
    def depth_rx(k):
        rx = r"[^()]*"
        for _ in range(k): rx = rf"(?:[^()]|\({rx}\))*"
        return rx
    def nested(d): return "(" * d + "x" + ")" * d
    OUT["task3"]["bracket"] = [dict(k=k, ok_depth=max(d for d in range(1, 9) if re.fullmatch(depth_rx(k), nested(d))) if re.fullmatch(depth_rx(k), nested(1)) else 0, fails_at=k + 1,
                                    rejects_deeper=re.fullmatch(depth_rx(k), nested(k + 1)) is None) for k in range(1, 6)]
    tk = sum(len(tokenize(t)) for t in texts[:50]); naive = sum(len(t.split()) for t in texts[:50])
    OUT["task3"]["token_counts_50rec"] = dict(maximal_munch=tk, whitespace=naive)

    # ---------------- Task 4 automata ----------------
    T4 = {}
    dfas = dfa_chain_specs()
    tests = {"MRN identifier": (["MRN-123456"], ["MRN-12345", "mrn-123456", "MRN123456"]), "ICD-10 code": (["I10", "E11.9", "K35.80"], ["U07", "I1", "E11."]),
             "Blood pressure": (["120/80", "95/60", "140/100"], ["1/80", "120/", "1200/80"]), "Date dd/mm/yyyy": (["03/04/2024", "3/4/2024"], ["03/04/24", "03-04-2024"])}
    T4["dfas"] = {k: dict(states=len(d.Q), sigma=d.sigma, table=d.table(), finals=sorted(d.F), edges=[[a, c, b] for (a, c), b in d.d.items()],
                          accepted_ok=all(d.accepts(x) for x in tests[k][0]), rejected_ok=not any(d.accepts(x) for x in tests[k][1]), tests=tests[k]) for k, d in dfas.items()}
    nfa, cl = temp_nfa()
    rep = {"D": "5", ".": ".", " ": " ", "U": "C"}
    for w in ["37.5 C", "37.5 K"]:
        rows, ok = nfa.trace(w); T4.setdefault("nfa_traces", []).append(dict(w=w, rows=rows, accepted=ok))
    d_, f_, rows = subset(nfa, rep)
    part, nmin, dead = minimise(d_, f_, len(rows), list(rep))
    T4["subset"] = dict(rows=[dict(dstate=r_[0], nfa=r_[1], moves=r_[2], final=r_[3]) for r_ in rows], dfa_states=len(rows), min_states=nmin, nfa_edges=[(a, l, sorted(b)) for (a, l), b in nfa.t.items()])
    lab = {"D": r"\d", ".": r"\.", " ": " ", "U": "[CF]"}
    inv = {}
    for st_, c in part.items():
        if st_ < len(rows): inv.setdefault(c, st_)
    md = {(part[a], c): part[b] for (a, c), b in d_.items()}
    dr = dfa_to_regex(sorted(set(part[q] for q in range(len(rows)))), md, part[0], {part[q] for q in f_}, lab)
    mdfa = DFA(md, {part[q] for q in f_}, cl, q0=part[0])
    T4["derived_regex"] = dr
    T4["equiv_nfa_dfa_regex"] = dict(rx_orig=r"\d\d(?:\.\d)? ?[CF]", mismatch_dfa=eq_test(lambda w: mdfa.accepts(w), r"\d\d(?:\.\d)? ?[CF]", "0123456789. CFK")[0],
                                     mismatch_derived=eq_test(lambda w: re.fullmatch(dr, w) is not None, r"\d\d(?:\.\d)? ?[CF]", "0123456789. CFK")[0], n=3000)
    th = {}
    for name, rx in (("dosage frequency", r"(OD|BD|TDS|QID|HS|SOS|PRN)"), ("contact number", r"(\+91-?)?[6-9]" + r"\d" * 9), ("temperature", r"\d\d(\.\d)? ?[CF]")):
        b = build_re_dfa(rx)
        th[name] = dict(regex=rx, nfa_states=b["nfa_states"], dfa_states=b["dfa_states"], min_states=b["min_states"], classes=len(b["reps"]),
                        mismatch=eq_test(b["dfa"].accepts, rx, "0123456789+-ODBTSQIHPRNCF. ", 3000)[0])
        if name == "dosage frequency": T4["thompson_rows"] = [dict(dstate=r_[0], nfa=r_[1], moves=r_[2], final=r_[3]) for r_ in b["rows"]][:14]
        if name == "contact number": PHONE_DFA = b["dfa"]
    T4["thompson"] = th
    T4["pumping"] = OUT["task3"]["bracket"]
    OUT["task4"] = T4

    # ---------------- Tasks 5-8 morphology ----------------
    forms = paradigm_forms()
    OUT["lexicon"]["paradigm_forms"] = len(forms)
    para = {}
    for lx in [("fracture", "V"), ("operate", "V"), ("patient", "N"), ("kidney", "N"), ("diagnosis", "N"), ("heavy", "ADJ"), ("give", "V"), ("study", "V")]:
        tags = {"N": ["SG", "PL"], "V": ["BASE", "3SG", "PAST", "PASTP", "PRESP"], "ADJ": ["BASE", "COMP", "SUP"]}[lx[1]]
        para[f"{lx[0]} ({lx[1]})"] = {t: FSTX.gen1(f"{lx[0]}+{lx[1]}+{t}") for t in tags}
    OUT["task5"] = dict(paradigms=para)
    def ovr(rule_name, lex_, expected):
        f = FST(LEX, IRR_V, IRR_N, IRR_A, use_exc=False); return dict(rule=rule_name, lexical=lex_, produced=f.gen1(lex_), expected=expected)
    OUT["task5"]["overgeneration"] = [ovr("e-deletion", "agree+V+PAST", "agreed"), ovr("e-deletion", "dye+V+PRESP", "dyeing"), ovr("consonant doubling", "visit+V+PAST", "visited"),
                                      ovr("consonant doubling", "vomit+V+PRESP", "vomiting"), ovr("consonant doubling", "poison+V+PAST", "poisoned"), ovr("sibilant -es", "stomach+N+PL", "stomachs"),
                                      ovr("y-to-i", "study+V+PRESP", "studying")]
    # ---- Task 7: traces & ambiguity ----
    def short(path): return dict(states=[path[0][0]] + [p[3] for p in path], steps=[(p[1] or "ε", p[2] or "ε") for p in path])
    T7 = {"parse_traces": {}, "gen_traces": {}, "ambiguous": {}}
    for sform in ["fractures", "went", "nephrectomies", "dressing", "readmitted"]:
        tr = FSTX.parse(sform, trace=True)
        T7["parse_traces"][sform] = [dict(lexical=l, **short(p)) for l, p in tr]
    for lx_ in ["fracture+N+PL", "go+V+PAST", "nephr+CF+ECTOMY+N+PL", "stop+V+PAST"]:
        T7["gen_traces"][lx_] = [dict(surface=sf, **short(p)) for sf, p in FSTX.gen(lx_, trace=True)]
    for sform in ["left", "normal", "saw", "dressing", "patient", "present", "discharged", "cough", "swollen", "antibiotic", "activity", "fractures"]:
        T7["ambiguous"][sform] = FSTX.parse(sform)
    T7["same_machine"] = dict(states=FSTX.n, arcs=FSTX.n_arcs(), roundtrip=[(l, FSTX.gen1(l), FSTX.parse(FSTX.gen1(l)).count(l) == 1) for l in ["fracture+N+PL", "go+V+PAST", "dizzy+ADJ+NESS+N+SG"]])
    T7["rejects_impossible"] = {l: FSTX.gen(l) for l in ["nephr+CF+ITIS+ECTOMY+N+SG", "ion+V+ATION+N+SG", "fracture+N+ITIS+N+SG", "hyper-hyper-glyc+CF+AEMIA+N+SG"]}
    OUT["task7"] = T7
    # ---- Task 8 ----
    TEST8 = [("fracture+V+PAST", "fractured"), ("prescribe+V+PRESP", "prescribing"), ("stop+V+PAST", "stopped"), ("admit+V+PRESP", "admitting"), ("refer+V+PAST", "referred"), ("occur+V+PASTP", "occurred"),
             ("visit+V+PAST", "visited"), ("vomit+V+PRESP", "vomiting"), ("body+N+PL", "bodies"), ("study+V+PAST", "studied"), ("heavy+ADJ+COMP", "heavier"), ("dry+ADJ+COMP", "drier"),
             ("dizzy+ADJ+NESS+N+SG", "dizziness"), ("box+N+PL", "boxes"), ("stitch+N+PL", "stitches"), ("rash+N+PL", "rashes"), ("diagnosis+N+PL", "diagnoses"), ("stomach+N+PL", "stomachs"),
             ("agree+V+PAST", "agreed"), ("dye+V+PRESP", "dyeing"), ("nephr+CF+ECTOMY+N+PL", "nephrectomies"), ("hyper-glyc+CF+AEMIA+N+SG", "hyperglycaemia"), ("re-admit+V+PASTP", "readmitted"),
             ("act+V+IVE+ADJ+ITY+N+SG", "activity"), ("measure+V+ABLE+ADJ+BASE", "measurable"), ("late+ADJ+SUP", "latest"), ("big+ADJ+SUP", "biggest"), ("go+V+PAST", "went"), ("give+V+PASTP", "given"),
             ("continue+V+PAST", "continued")]
    F_noexc = FST(LEX, IRR_V, IRR_N, IRR_A, use_exc=False); F_inter = FST(LEX, IRR_V, IRR_N, IRR_A, rules=())
    rows8 = []
    for lx_, exp in TEST8:
        got = FSTX.gen1(lx_); got0 = F_noexc.gen1(lx_); inter = (F_inter.gen1(lx_) or "").replace("#", "")
        rows8.append(dict(lexical=lx_, intermediate=inter, expected=exp, cascade=got, cascade_no_exc=got0, parallel=parallel(RULES, (F_inter.gen1(lx_) or "")) if inter else None))
    OUT["task8"] = dict(rules=[r.text() for r in RULES], names=[r.name for r in RULES], tests=rows8,
                        pass_cascade=sum(r["cascade"] == r["expected"] for r in rows8), pass_noexc=sum(r["cascade_no_exc"] == r["expected"] for r in rows8),
                        pass_parallel=sum(r["parallel"] == r["expected"] for r in rows8), disagree_cascade_parallel=[r["lexical"] for r in rows8 if r["cascade"] != r["parallel"]])
    same = 0; first5 = list(RULES[:5]); orders = list(itertools.permutations(first5))
    good_orders = []
    for pm in orders:
        F = FST(LEX, IRR_V, IRR_N, IRR_A, rules=tuple(pm) + (R_CLEAN,))
        ok = sum(F.gen1(l) == FSTX.gen1(l) for l, _ in TEST8)
        if ok == len(TEST8): good_orders.append([r.name for r in pm])
    OUT["task8"]["orderings"] = dict(total=len(orders), identical_output=len(good_orders), example_good=good_orders[:3])
    print("task8 cascade pass", OUT["task8"]["pass_cascade"], "/", len(TEST8), "no-exc", OUT["task8"]["pass_noexc"], "parallel", OUT["task8"]["pass_parallel"], "orders ok", len(good_orders), "/", len(orders))

    # ---------------- gold lemma bookkeeping ----------------
    lem_by_form = defaultdict(Counter); catof = {}
    for r in recs:
        for surf, lem, ft, cat, lang in r.lem:
            if cat != "typo": lem_by_form[surf][(lem, ft)] += 1; catof[surf] = cat
    surf_by_lem = defaultdict(set)
    for f_, c in lem_by_form.items():
        for (lem, ft) in c: surf_by_lem[lem].add(f_)
    gold_lemmas = {f_: c.most_common(1)[0][0][0] for f_, c in lem_by_form.items()}
    # Task 5 stats
    known_types = {w for w in types if w in gold_lemmas}
    OUT["task5"].update(types_known=len(known_types), lemmas=len({gold_lemmas[w] for w in known_types}), ratio=round(len(known_types) / len({gold_lemmas[w] for w in known_types}), 3),
                        irregular_forms_in_corpus=sorted(f_ for f_, c in catof.items() if c == "irr")[:60], n_irregular_types=sum(1 for c in catof.values() if c == "irr"))
    # ---------------- retrieval experiment (Tasks 9, 11) ----------------
    gold_docs = [set(l[1] for l in r.lem) for r in recs]
    df = Counter(l for g in gold_docs for l in g)
    forced = W("operate operation observe observation treat treatment administer ventilate ventilation fracture bleed leave give diagnose diagnosis nephrectomy hepatitis infect infection medic")
    singles = [q for q in forced if df[q] >= 5][:14]
    pool = [q for q, c in sorted(df.items()) if 8 <= c <= 160 and q not in singles and q.isalpha()]
    R.shuffle(pool); singles += pool[:40 - 14 - 12 + (14 - len(singles))]
    pairs = []
    for _ in range(4000):
        a, b = R.sample(sorted(df), 2)
        if a != b and df[a] >= 20 and df[b] >= 20 and sum(1 for g in gold_docs if a in g and b in g) >= 3 and (a, b) not in pairs: pairs.append((a, b))
        if len(pairs) == 12: break
    queries = [(q,) for q in singles[:28]] + pairs
    queries = queries[:40]
    OUT["task9"] = dict(n_queries=len(queries), queries=[" + ".join(q) for q in queries])
    toks_per_doc = [[w.lower() for w in words_of(t)] for t in texts]
    cond = {"none": [set(ts) for ts in toks_per_doc], "porter": [{PS.stem(w) for w in ts} for ts in toks_per_doc], "fst": [set().union(*[lem_terms(FSTX, w) for w in ts]) for ts in toks_per_doc]}
    qmap = {"none": lambda q: q, "porter": PS.stem, "fst": lambda q: q}
    metrics = {}
    for c in cond:
        TP = FP = FN = 0
        for q in queries:
            rel = {i for i, g in enumerate(gold_docs) if all(x in g for x in q)}
            ret = {i for i, d in enumerate(cond[c]) if all(qmap[c](x) in d for x in q)}
            TP += len(rel & ret); FP += len(ret - rel); FN += len(rel - ret)
        p, r_ = TP / max(1, TP + FP), TP / max(1, TP + FN)
        metrics[c] = dict(tp=TP, fp=FP, fn=FN, p=round(p, 3), r=round(r_, 3), f1=round(2 * p * r_ / max(1e-9, p + r_), 3))
    OUT["task9"]["retrieval"] = metrics
    # inflection-only losses
    miss = irr_only = 0
    for q in queries:
        for i, g in enumerate(gold_docs):
            if all(x in g for x in q) and not all(x in cond["none"][i] for x in q):
                miss += 1
                forms_here = {l[0] for l in recs[i].lem if l[1] in q}
                irr_only += all(catof.get(f_) == "irr" for f_ in forms_here)
    nrel = sum(1 for q in queries for g in gold_docs if all(x in g for x in q))
    OUT["task5"]["inflection_effect"] = dict(relevant_pairs=nrel, missed_without_normalisation=miss, share=round(miss / nrel, 3), irregular_only_among_missed=irr_only)
    # ---------------- Task 9: Porter ----------------
    def mform(w):
        cv = "".join("V" if (c in "aeiou" or (c == "y" and i > 0 and w[i - 1] not in "aeiou")) else "C" for i, c in enumerate(w))
        f = re.sub(r"(.)\1+", r"\1", cv); return f, f.count("VC")
    def steps(w):
        out, cur = [], w
        for nm in ("1a", "1b", "1c", "2", "3", "4", "5a", "5b"):
            new = getattr(PS, "_step" + nm)(cur)
            if new != cur: out.append((nm, cur, new))
            cur = new
        return cur, out
    OUT["task9"]["traces"] = []
    for w in ["fractures", "operations", "hypertensive", "nephrectomies", "bleeding", "dizziness", "inflammation", "transfusions"]:
        f_, m_ = mform(w); fin, st = steps(w)
        OUT["task9"]["traces"].append(dict(word=w, form=f_, m=m_, stem=fin, steps=st, nltk=PS.stem(w), has_vowel=bool(re.search("[aeiou]", w[:-3])), double_c=bool(re.search(r"([^aeiou])\1$", w))))
    # annotated forms
    lem_cat = defaultdict(list)
    for lem, fs in surf_by_lem.items():
        c = {catof[f_] for f_ in fs}
        lem_cat["irr" if "irr" in c else "der" if "der" in c else "reg"].append(lem)
    chosen = []
    for cat, nlem in (("irr", 28), ("der", 40), ("reg", 200)):
        pl = [l for l in sorted(lem_cat[cat]) if len(surf_by_lem[l]) >= (1 if cat == "irr" else 2)]; R.shuffle(pl)
        for l in pl[:nlem]:
            for f_ in sorted(surf_by_lem[l])[:4]:
                if len(chosen) < 200 and (f_, l) not in chosen and gold_lemmas.get(f_) == l: chosen.append((f_, l))
    annot = [(f_, l, catof[f_]) for f_, l in chosen]
    agree = Counter(); rows9 = []
    amb_n = 0
    for f_, l, cat in annot:
        an = FSTX.parse(f_); lems = [lexical_to_lemma(x) for x in an]
        amb_n += len(set(lems)) > 1
        fst_first = lems[0] if lems else None
        fst_ok = fst_first == l; fst_or = l in lems
        p_ok = PS.stem(f_) == PS.stem(l)
        agree[(cat, p_ok, fst_ok)] += 1
        rows9.append((f_, l, cat, PS.stem(f_), fst_first, p_ok, fst_ok, fst_or))
    ov = ovn = un = unn = fov = fun = 0
    for (f1, l1, *_), (f2, l2, *_) in itertools.combinations(rows9, 2):
        same_l = l1 == l2; same_s = PS.stem(f1) == PS.stem(f2)
        if same_l: unn += 1; un += not same_s
        else: ovn += 1; ov += same_s
    OUT["task9"]["annot"] = dict(n=len(annot), ambiguous=amb_n, by_cat=dict(Counter(c for *_, c in annot)),
                                 porter_conflates_with_lemma=sum(r[5] for r in rows9), fst_first_correct=sum(r[6] for r in rows9), fst_oracle_correct=sum(r[7] for r in rows9),
                                 agreement={f"{k[0]}|porter={k[1]}|fst={k[2]}": v for k, v in agree.items()},
                                 over_pairs=ov, over_den=ovn, over_rate=round(ov / ovn, 5), under_pairs=un, under_den=unn, under_rate=round(un / max(1, unn), 4),
                                 sample=[r[:5] for r in rows9 if (not r[5]) or (not r[6])][:14],
                                 over_examples=[(a[0], b[0], PS.stem(a[0])) for a, b in itertools.combinations(rows9, 2) if a[1] != b[1] and PS.stem(a[0]) == PS.stem(b[0])][:10],
                                 under_examples=[(a[0], b[0], PS.stem(a[0]), PS.stem(b[0])) for a, b in itertools.combinations(rows9, 2) if a[1] == b[1] and PS.stem(a[0]) != PS.stem(b[0])][:10])
    # vocabulary-level stemming errors over every form the lexicon can generate (Porter stem classes vs FST lemmas)
    vl = {f_: lexical_to_lemma(FSTX.parse(f_)[0]) for f_ in forms if FSTX.parse(f_)}
    sc, lc = defaultdict(set), defaultdict(set)
    for f_, l in vl.items(): sc[PS.stem(f_)].add(l); lc[l].add(PS.stem(f_))
    OUT["task9"]["vocab_stemming"] = dict(forms=len(vl), lemmas=len(lc), stem_classes=len(sc), over_classes=sum(len(v) > 1 for v in sc.values()), under_lemmas=sum(len(v) > 1 for v in lc.values()),
                                          over_examples=[(k, sorted(v)[:4]) for k, v in sc.items() if len(v) > 1][:12], under_examples=[(k, sorted(v)[:4]) for k, v in lc.items() if len(v) > 1][:12])
    # ---------------- Task 6 productivity ----------------
    prod = {}
    tokc = Counter(w.lower() for t in texts for w in words_of(t))
    for ty in tokc:
        for lx_ in FSTX.parse(ty):
            for tg in re.findall(r"\+(ITIS|ECTOMY|OTOMY|OLOGY|OSIS|AEMIA|IC|AL|ATION|ION|ABLE|IVE|MENT|ITY|NESS|LY)(?=\+)", lx_):
                prod.setdefault(tg, Counter())[ty] += tokc[ty]
            for pf in re.findall(r"^([a-z]+)-", lx_): prod.setdefault(pf + "-", Counter())[ty] += tokc[ty]
    pr_rows = []
    for k, c in sorted(prod.items(), key=lambda x: -sum(x[1].values())):
        n1 = sum(1 for v in c.values() if v == 1); N_ = sum(c.values())
        pr_rows.append(dict(affix=k, types=len(c), tokens=N_, hapax=n1, P=round(n1 / N_, 4), cls="productive" if len(c) >= 10 else "limited"))
    OUT["task6"] = dict(affixes=pr_rows, n_affixes_attested=len(pr_rows),
                        families={r: [f_ for f_ in sorted(forms) if forms[f_] == r][:8] for r in ["nephr", "hepat", "cardi", "operate", "treat", "toxic"]})
    # ---------------- pipeline stats (Task 10) ----------------
    stage = []
    tt = time.perf_counter(); alltoks = [tokenize(t) for t in texts]; dt = time.perf_counter() - tt
    ntok = sum(len(x) for x in alltoks); stage.append(dict(stage="Tokenizer (maximal munch)", items=len(texts), produced=ntok, rejected=sum(1 for x in alltoks for n, _ in x if n == "PUNCT"), sec=round(dt, 3)))
    tt = time.perf_counter(); ents = [find_entities(t) for t in texts]; dt = time.perf_counter() - tt
    nen = sum(len(x) for x in ents); stage.append(dict(stage="Entity regular expressions (12 types)", items=len(texts), produced=nen, rejected=0, sec=round(dt, 3)))
    tt = time.perf_counter(); acc = rej = 0
    for x in ents:
        for k, v in x:
            if k in ("PID", "ICD", "BP") or (k == "DATE" and "/" in v) or k == "PHONE":
                ok = dfas[{"PID": "MRN identifier", "ICD": "ICD-10 code", "BP": "Blood pressure", "DATE": "Date dd/mm/yyyy"}[k]].accepts(v.replace(" mmHg", "")) if k != "PHONE" else PHONE_DFA.accepts(v.replace("-", "-"))
                acc += ok; rej += not ok
    dt = time.perf_counter() - tt; stage.append(dict(stage="Automaton validators (PID, ICD, BP, date, phone)", items=acc + rej, produced=acc, rejected=rej, sec=round(dt, 3)))
    wt = [w.lower() for t in texts for w in words_of(t)]
    tt = time.perf_counter(); FSTX.cache.clear(); an = [FSTX.parse(w) for w in wt]; dt = time.perf_counter() - tt
    stage.append(dict(stage="Morphological transducer (parse)", items=len(wt), produced=sum(1 for a in an if a), rejected=sum(1 for a in an if not a), sec=round(dt, 3)))
    tt = time.perf_counter(); st_ = [PS.stem(w) for w in wt]; dt = time.perf_counter() - tt
    stage.append(dict(stage="Porter stemmer", items=len(wt), produced=len(st_), rejected=0, sec=round(dt, 3)))
    OUT["task10"] = dict(stages=stage, coverage_tokens=round(sum(1 for a in an if a) / len(wt), 4), coverage_types=round(sum(1 for w in types if FSTX.parse(w)) / len(types), 4))
    unk = Counter(w for w, a in zip(wt, an) if not a)
    OUT["task10"]["top_oov"] = unk.most_common(15)
    OUT["task10"]["ambiguous_types"] = dict(n=sum(1 for w in types if len(FSTX.parse(w)) > 1), of=len(types))
    # unit tests
    ut = []
    exp_hand = {"nephrectomies": "nephr+CF+ECTOMY+N+PL", "went": "go+V+PAST", "fractures": None, "left": None}
    ut.append(("DFA accept/reject cases", all(v["accepted_ok"] and v["rejected_ok"] for v in T4["dfas"].values())))
    ut.append(("NFA temperature: '37.5 C' accepted, '37.5 K' rejected", T4["nfa_traces"][0]["accepted"] and not T4["nfa_traces"][1]["accepted"]))
    ut.append(("Subset/min DFA = regex on 3000 random strings", T4["equiv_nfa_dfa_regex"]["mismatch_dfa"] == 0 and T4["equiv_nfa_dfa_regex"]["mismatch_derived"] == 0))
    ut.append(("Thompson DFAs = re.fullmatch on 3000 random strings (x3)", all(v["mismatch"] == 0 for v in th.values())))
    ut.append(("Irregulars parse (went, gave, swollen, feet, diagnoses)", all(FSTX.parse(x) for x in ["went", "gave", "swollen", "feet", "diagnoses"])))
    ut.append(("Hand analysis nephrectomies = nephr+CF+ECTOMY+N+PL", FSTX.parse("nephrectomies") == ["nephr+CF+ECTOMY+N+PL"]))
    ut.append(("Ambiguity: fractures has N and V readings", len(FSTX.parse("fractures")) == 2))
    ut.append(("Ambiguity: left has 3 analyses", len(FSTX.parse("left")) == 3))
    ut.append(("Porter hand trace matches NLTK for 8 words", all(t["stem"] == t["nltk"] for t in OUT["task9"]["traces"])))
    ut.append(("Rejects: every pattern has a negative example", all(not re.search(PAT[k], x) for k, x in [("PID", "MRN-12345"), ("DATE", "13/45"), ("DRUG", "paracetamol 500 mg"), ("FREQ", "ODD"), ("LAB", "5 kg"), ("BP", "12/8"), ("TEMP", "5 C"),
                                                                                                                   ("ICD", "U07"), ("WARD", "Ward 4"), ("PHONE", "5123456789"), ("AGE", "45 kg"), ("DUR", "3 apples")])))
    ut.append(("Parse(gen(x)) recovers x for 3 forms", all(r[2] for r in T7["same_machine"]["roundtrip"])))
    OUT["task10"]["unit_tests"] = ut
    # plot data: coverage vs lexicon size; runtime vs length
    cov = []
    ents_list = list(LEX.items()); tokc_all = Counter(wt)
    for frac in (.2, .4, .6, .8, 1.0):
        R2 = random.Random(3); keep = {k: v for k, v in ents_list if k[1] == "FUNC" or R2.random() < frac}
        F = FST(keep, IRR_V, IRR_N, IRR_A); cov.append((frac, len([1 for k in keep if k[1] != "FUNC"]), round(sum(c for w, c in tokc_all.items() if F.parse(w)) / len(wt), 4)))
    rt = []
    for k in (1, 5, 10, 20, 40, 80):
        big = " ".join(texts[:k]); tt = time.perf_counter()
        for _ in range(3): tokenize(big); find_entities(big)
        rt.append((k, len(big.split()), round((time.perf_counter() - tt) / 3, 4)))
    OUT["task10"]["coverage_curve"], OUT["task10"]["runtime_curve"] = cov, rt
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
        ax[0].plot([c[1] for c in cov], [c[2] * 100 for c in cov], "o-"); ax[0].set_xlabel("lexicon stems"); ax[0].set_ylabel("token coverage (%)"); ax[0].set_title("Analysis coverage vs lexicon size")
        ax[1].plot([r_[1] for r_ in rt], [r_[2] * 1000 for r_ in rt], "s-", color="tab:red"); ax[1].set_xlabel("record length (whitespace words)"); ax[1].set_ylabel("ms"); ax[1].set_title("Tokenize+patterns run-time")
        plt.tight_layout(); plt.savefig("coverage.png", dpi=150)
    except Exception as e: print("plot skipped:", e)

    # ---------------- Task 12: noisy input ----------------
    vocab = paradigm_forms(); inv_big = defaultdict(set)
    for f_ in vocab:
        for g in set(ngrams(f_)): inv_big[g].add(f_)
    tav = {v: k for k, vs in TAVAR.items() for v in vs}; tav.update({k: k for k in TAMIL}); tav.update({w: w for w in TAFUNC})
    lid = LangID(list(vocab), list(tav))
    def nearest(w):
        votes = Counter(f_ for g in set(ngrams(w)) for f_ in inv_big.get(g, ()) if abs(len(f_) - len(w)) <= 2)
        best = min(((wed(w, f_), f_) for f_, _ in votes.most_common(12)), default=(9, None)); return best
    stat = Counter()
    def revised_terms(text):
        t = repair_ocr(text); terms = set(); words = [w.lower() for w in words_of(t)]
        for w in words:
            stat["tokens"] += 1
            a = FSTX.parse(w)
            if a: terms |= {lexical_to_lemma(x) for x in a}; stat["direct"] += 1; continue
            if len(w) < 3: stat["short"] += 1; continue
            lang = lid.predict(w)
            if lang == "ta":
                c, bt = min(((wed(w, v), v) for v in tav), default=(9, None))
                if c <= .4 * len(w):
                    stat["tamil"] += 1; g = TAMIL.get(tav[bt])
                    if g: terms.add(g)
                    continue
            c, f_ = nearest(w)
            if f_ and c <= .9 and len(w) >= 4:
                stat["wed_fix"] += 1; terms |= {lexical_to_lemma(x) for x in FSTX.parse(f_)}; continue
            stat["fallback"] += 1; terms.add("~" + PS.stem(w))
        return terms
    sets = {}
    for name, mk in (("ocr", lambda: make_record(pick(list(TEMPL)))), ("codemixed", code_mixed_record), ("both", code_mixed_record)):
        sets[name] = []
        for _ in range(50):
            r = mk(); tx = r.text()
            if name != "codemixed": tx = ocr_noise(tx)
            sets[name].append((r, tx))
    LIDacc = [0, 0, 0, 0]
    for r, tx in sets["codemixed"]:
        for surf, lem, ft, cat, lang in r.lem:
            if cat == "typo" or not surf.isalpha() or len(surf) < 3: continue
            pr = lid.predict(surf.lower()); LIDacc[0] += pr == lang; LIDacc[1] += 1
            if lang == "ta": LIDacc[2] += pr == "ta"; LIDacc[3] += 1
    T12 = dict(lid=dict(accuracy=round(LIDacc[0] / LIDacc[1], 3), tamil_recall=round(LIDacc[2] / max(1, LIDacc[3]), 3), n=LIDacc[1]), samples={})
    base_gold = [set(l[1] for l in r.lem) for r in recs]
    for name, items in sets.items():
        gold_ents = [r.ents for r, _ in items]; gdocs = [set(l[1] for l in r.lem) for r, _ in items]
        res_ = {}
        for variant in ("original", "revised"):
            tt = time.perf_counter(); stat.clear(); TP = FP = FN = 0; docs = []; direct = ntok_ = 0
            for (r, tx), ge in zip(items, gold_ents):
                tx2 = repair_ocr(tx) if variant == "revised" else tx
                a, b, c = prf(find_entities(tx2), ge); TP += a; FP += b; FN += c
                if variant == "original":
                    ws = [w.lower() for w in words_of(tx)]; docs.append(set().union(*[lem_terms(FSTX, w) for w in ws])); direct += sum(1 for w in ws if FSTX.parse(w)); ntok_ += len(ws)
                else: docs.append(revised_terms(tx))
            dt = time.perf_counter() - tt
            qTP = qFP = qFN = 0
            for q in queries:
                rel = {i for i, g in enumerate(gdocs) if all(x in g for x in q)}
                ret = {i for i, d in enumerate(docs) if all((x in d) or ("~" + PS.stem(x) in d) for x in q)} if variant == "revised" else {i for i, d in enumerate(docs) if all(x in d for x in q)}
                qTP += len(rel & ret); qFP += len(ret - rel); qFN += len(rel - ret)
            qp, qr = qTP / max(1, qTP + qFP), qTP / max(1, qTP + qFN)
            cov_ = (direct / max(1, ntok_)) if variant == "original" else (stat["direct"] + stat["wed_fix"] + stat["tamil"]) / max(1, stat["tokens"])
            res_[variant] = dict(ent_p=round(TP / max(1, TP + FP), 3), ent_r=round(TP / max(1, TP + FN), 3), ret_p=round(qp, 3), ret_r=round(qr, 3), ret_f1=round(2 * qp * qr / max(1e-9, qp + qr), 3),
                                 coverage=round(cov_, 3), sec=round(dt, 2), stats=dict(stat) if variant == "revised" else {})
        T12["samples"][name] = res_
    r0, tx0 = sets["ocr"][0]; r1, tx1 = sets["codemixed"][1]; r2, tx2 = sets["both"][2]
    T12["traces"] = []
    for nm, (r, tx) in (("OCR", (r0, tx0)), ("Code-mixed", (r1, tx1)), ("OCR + code-mixed", (r2, tx2))):
        ws = [w.lower() for w in words_of(tx)]
        T12["traces"].append(dict(kind=nm, text=tx[:330], gold_entities=r.ents[:6], found=find_entities(tx)[:6], oov=[w for w in ws if not FSTX.parse(w)][:14], repaired=repair_ocr(tx)[:200]))
    T12["clean_baseline"] = dict(ent_p=OUT["task2"]["micro_p"], ent_r=OUT["task2"]["micro_r"], ret_f1=metrics["fst"]["f1"], coverage=OUT["task10"]["coverage_tokens"])
    T12["oov_share_noisy"] = {n: round(1 - T12["samples"][n]["original"]["coverage"], 3) for n in sets}
    OUT["task12"] = T12
    OUT["runtime_sec"] = round(time.time() - t0, 1)
    json.dump(OUT, open("results.json", "w"), indent=1, default=str)
    print("\n=== SUMMARY ===")
    print("retrieval", metrics); print("annot", {k: v for k, v in OUT["task9"]["annot"].items() if k in ("n", "over_rate", "under_rate", "porter_conflates_with_lemma", "fst_first_correct", "fst_oracle_correct")})
    for s_ in stage: print(s_)
    print("unit tests:", [(n, ok) for n, ok in ut]); print("task12", json.dumps({k: v for k, v in T12["samples"].items()}, indent=0)[:1500]); print("lid", T12["lid"], "time", OUT["runtime_sec"])


if __name__ == "__main__":
    main()

corpus: 520 records, 36280 word tokens, 1709 alphabetic types
lexicon: 511 stems, 3370 FST states, 5202 arcs
task2 micro P/R 0.865 0.942
task8 cascade pass 29 / 30 no-exc 25 parallel 29 orders ok 30 / 120

=== SUMMARY ===
retrieval {'none': {'tp': 247, 'fp': 66, 'fn': 327, 'p': 0.789, 'r': 0.43, 'f1': 0.557}, 'porter': {'tp': 511, 'fp': 299, 'fn': 63, 'p': 0.631, 'r': 0.89, 'f1': 0.738}, 'fst': {'tp': 561, 'fp': 77, 'fn': 13, 'p': 0.879, 'r': 0.977, 'f1': 0.926}}
annot {'n': 200, 'porter_conflates_with_lemma': 170, 'fst_first_correct': 199, 'fst_oracle_correct': 200, 'over_rate': 0.0003, 'under_rate': 0.0965}
{'stage': 'Tokenizer (maximal munch)', 'items': 520, 'produced': 45069, 'rejected': 8789, 'sec': 0.178}
{'stage': 'Entity regular expressions (12 types)', 'items': 520, 'produced': 5755, 'rejected': 0, 'sec': 0.123}
{'stage': 'Automaton validators (PID, ICD, BP, date, phone)', 'items': 1760, 'produced': 1760, 'rejected': 0, 'sec': 0.005}
{'stage': 'Morphological transducer (parse)